In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
%sql
show tables in workspace.bronze

## Access the data


In [0]:
df_aisles=spark.table('workspace.bronze.bronze_aisles')

In [0]:
df_departments=spark.table('workspace.bronze.bronze_departments')
df_orders=spark.table('workspace.bronze.bronze_orders')
df_order_products_prior=spark.table('workspace.bronze.bronze_order_products_prior')
df_products=spark.table('workspace.bronze.bronze_products')
df_order_products_train=spark.table('workspace.bronze.bronze_order_products_train')

In [0]:
df_departments.printSchema()

In [0]:
df_aisles.printSchema()

### change the column Type

In [0]:
df_orders = df_orders.withColumn(
    "days_since_prior_order", 
    col("days_since_prior_order").cast("integer")
)

In [0]:
df_orders.printSchema()

In [0]:
df_products=df_products.withColumn('aisle_id',col("aisle_id").cast("integer"))\
    .withColumn('department_id',col("department_id").cast("integer"))

In [0]:
df_products.printSchema()

In [0]:
print("count of df_asiles : ",df_aisles.count())
print("count of df_departments : ",df_departments.count())
print("count of df_orders : ",df_orders.count())
print("count of df_products_prior : ",df_order_products_prior.count())
print("count of df_products : ",df_products.count())
print("count of df_products_train : ",df_order_products_train.count())

In [0]:
df_orders.select(["order_id",'user_id',"eval_set","order_number","order_dow","order_hour_of_day","days_since_prior_order"]).dropDuplicates(["order_id"]).count()

In [0]:
df_orders=df_orders.select(["order_id",'user_id',"eval_set","order_number","order_dow","order_hour_of_day","days_since_prior_order"]).dropDuplicates(["order_id"])

### create schema

In [0]:
%sql
create schema if not exists workspace.silver

In [0]:
df_orders.write.format('delta').mode("overwrite").saveAsTable("workspace.silver.silver_orders")


In [0]:
df_order_products_prior=df_order_products_prior.select(["order_id","product_id","add_to_cart_order","reordered"]).withColumn("order_type", lit("prior"))

In [0]:
df_order_products_train=df_order_products_train.select(["order_id","product_id","add_to_cart_order","reordered"]).withColumn("order_type", lit("train"))

In [0]:
df_order_products_Mix=df_order_products_prior.unionByName(df_order_products_train)\
    .dropDuplicates(["order_id","product_id"])

In [0]:
df_order_products_Mix.write.format('delta').mode("overwrite").saveAsTable("workspace.silver.silver_order_products_Mix")

In [0]:
df_products.limit(1).display()

In [0]:
df_products.printSchema()

In [0]:
from pyspark.sql.functions import col, expr, trim

df_products_k = (
    spark.table("workspace.bronze.bronze_products")
    .withColumn("product_id", expr("try_cast(product_id as int)"))
    .withColumn("aisle_id", expr("try_cast(aisle_id as int)"))
    .withColumn("department_id", expr("try_cast(department_id as int)"))
    .withColumn("product_name", trim(col("product_name")))
    .select("product_id", "product_name", "aisle_id", "department_id")
    .dropDuplicates(["product_id"])
)

df_products_k.display()

In [0]:
df_products_k.printSchema()

In [0]:
df_products_k.write.format('delta').mode("overwrite").saveAsTable("workspace.silver.silver_products")

In [0]:
df_aisles.printSchema()

In [0]:
df_aisles=df_aisles.select("aisle_id",trim(col("aisle")).alias("aisle")).dropDuplicates(["aisle_id"])
df_aisles.display()

In [0]:
df_aisles.write.format('delta').mode('overwrite')\
    .saveAsTable("workspace.silver.silver_aisles")


In [0]:
df_departments.printSchema()



In [0]:
df_departments=df_departments.select("department_id",trim(col("department")).alias("department")).dropDuplicates(["department_id"])
df_departments.display()

In [0]:
df_departments.write.format('delta').mode('overwrite')\
    .saveAsTable("workspace.silver.silver_departments")

### check the Bronze Schema

In [0]:
%sql
show tables in workspace.bronze

### check the silver_schema

In [0]:
%sql
show tables in workspace.silver

In [0]:
print("count of df_asiles : ",df_aisles.count())
print("count of df_departments : ",df_departments.count())
print("count of df_orders : ",df_orders.count())
print("count of df_products_mix : ",df_order_products_Mix.count())
print("count of df_products : ",df_products.count())